# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aamr8010/flyrank-ml-internship/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
import pandas as pd
import numpy as np
import os

from datasets import load_dataset
from google.colab import userdata

# Load Hugging Face token
HF_TOKEN = userdata.get("HF_TOKEN")

if not HF_TOKEN:
    raise ValueError(
        "HF_TOKEN not found. Go to Colab -> Secrets -> make sure HF_TOKEN exists and is enabled."
    )

print("HF token loaded:", True)

# Load the same 30,000-row dataset used in previous weeks
ds = load_dataset(
    "FlyRank/internship-warehouse",
    "fact_content_daily_performance",
    split="train",
    streaming=True,
    token=HF_TOKEN
)

df = pd.DataFrame(list(ds.take(30000)))

print("Dataset loaded successfully.")
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())

HF token loaded: True


README.md:   0%|          | 0.00/3.04k [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/18 [00:00<?, ?it/s]

Dataset loaded successfully.
Shape: (30000, 30)
Columns: ['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events']


## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*



The playbook ranks content observations using simple, interpretable signals.

The goal is decision-support: identify observations that may deserve human review rather than automatically changing content.

### Reason codes

- REFRESH_REVIEW — older content that may deserve a freshness review.
- SEARCH_OPPORTUNITY — visible search activity with relatively weak engagement.
- ENGAGEMENT_REVIEW — measurable traffic with weaker engagement signals.
- MONITOR — insufficient evidence for a stronger action.

### Action mapping

- REFRESH_REVIEW → review freshness, relevance, and content accuracy.
- SEARCH_OPPORTUNITY → review search intent, title, structure, and SERP alignment.
- ENGAGEMENT_REVIEW → review page experience and content engagement.
- MONITOR → continue observing before taking action.

The score is a prioritization score, not a prediction of guaranteed business value.

## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*


### Intended use

This playbook is intended to support human prioritization of content review.

It can help a reviewer decide which observations deserve attention first based on observed search and engagement signals.

The score is directional and should be used as decision-support rather than as an automatic content decision.

### Limits

The score does not prove that refreshing content will improve performance.

The available anonymized data does not provide enough evidence to establish causal impact.

The ranking may reflect differences in traffic, search demand, content type, client history, or other factors that are not fully represented by the rule.

The output should therefore be reviewed by a human before any content change is made.

In [2]:
# Work on a copy
work = df.copy()

# Convert numeric fields safely
numeric_cols = [
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "ga4_sessions",
    "scroll_events"
]

for col in numeric_cols:
    work[col] = pd.to_numeric(work[col], errors="coerce")

work[numeric_cols] = work[numeric_cols].fillna(0)

# Normalize signals using percentile ranks
work["impression_score"] = work["gsc_impressions"].rank(pct=True)
work["session_score"] = work["ga4_sessions"].rank(pct=True)

# Lower position number is better, so reverse it
position_rank = work["gsc_avg_position"].rank(pct=True)
work["position_opportunity"] = 1 - position_rank

# Simple interpretable prioritization score
work["action_score"] = (
    0.40 * work["impression_score"]
    + 0.35 * work["position_opportunity"]
    + 0.25 * work["session_score"]
)

# Assign reason codes
def assign_reason(row):
    if row["gsc_impressions"] > work["gsc_impressions"].quantile(0.75):
        if row["gsc_avg_position"] > 10:
            return "SEARCH_OPPORTUNITY"

    if row["gsc_avg_position"] > 10:
        return "REFRESH_REVIEW"

    if row["ga4_sessions"] > work["ga4_sessions"].quantile(0.75):
        return "ENGAGEMENT_REVIEW"

    return "MONITOR"

work["reason_code"] = work.apply(assign_reason, axis=1)

# Map reason code to action
action_map = {
    "REFRESH_REVIEW": "Review and refresh",
    "SEARCH_OPPORTUNITY": "Review search alignment",
    "ENGAGEMENT_REVIEW": "Review engagement",
    "MONITOR": "Monitor"
}

work["action"] = work["reason_code"].map(action_map)

# Rank
work = work.sort_values(
    "action_score",
    ascending=False
).reset_index(drop=True)

work["rank"] = np.arange(1, len(work) + 1)

print("Ranked queue created.")
display(
    work[
        [
            "rank",
            "client_hash_id",
            "content_hash_id",
            "action_score",
            "reason_code",
            "action"
        ]
    ].head(20)
)

Ranked queue created.


,rank,client_hash_id,content_hash_id,action_score,reason_code,action
0,1,client_9958f0a7ae1df715,content_213eb91f21a43550,0.873086,MONITOR,Monitor
1,2,client_9958f0a7ae1df715,content_79bfcf05bc81bf82,0.872594,MONITOR,Monitor
2,3,client_9958f0a7ae1df715,content_f94fe855380e150f,0.872548,MONITOR,Monitor
3,4,client_73cda7b4e4f265ea,content_690b092cf66bc2a4,0.872539,MONITOR,Monitor
4,5,client_9958f0a7ae1df715,content_f94fe855380e150f,0.872466,MONITOR,Monitor
5,6,client_73cda7b4e4f265ea,content_690b092cf66bc2a4,0.872463,MONITOR,Monitor
6,7,client_73cda7b4e4f265ea,content_690b092cf66bc2a4,0.872351,MONITOR,Monitor
7,8,client_9958f0a7ae1df715,content_79bfcf05bc81bf82,0.872286,MONITOR,Monitor
8,9,client_9958f0a7ae1df715,content_f94fe855380e150f,0.872261,MONITOR,Monitor
9,10,client_9958f0a7ae1df715,content_f94fe855380e150f,0.872241,MONITOR,Monitor


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*



Every ranked recommendation requires human review before action.

### Human review checklist

Before applying an action, the reviewer should check:

1. Is the content still relevant to the intended search intent?
2. Is the information accurate and current?
3. Does the observed search position support the suggested action?
4. Are there signs that the signal is caused by an unusual event?
5. Is there enough evidence to justify spending time on the content?

### What should NOT be automated

The system should not automatically:

- rewrite or publish content;
- delete content;
- change factual claims;
- change search intent;
- make client-facing recommendations without review;
- declare that a content refresh will improve rankings;
- treat the score as a guaranteed business-value prediction.

## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*



The playbook should be monitored periodically rather than treated as a permanent rule.

### Monitoring

Track:

- distribution of action scores;
- number of items receiving each reason code;
- changes in search and engagement signal distributions;
- reviewer disagreement with recommendations;
- observed outcomes after reviewed actions.

### Review trigger

The rule should be reviewed if the distribution of recommendations changes substantially or if reviewers repeatedly reject the same recommendation type.

### Retrain / redesign trigger

A future model or rule should be considered when:

- the available data changes materially;
- new reliable outcome labels become available;
- validation performance degrades;
- important signals change meaning;
- reviewer feedback shows systematic errors.

These are monitoring and review triggers, not claims about production performance.

## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

## Cost / value thinking

The highest-ranked item should not automatically receive the highest-cost intervention.

A practical approach is:

- Low-cost review → check freshness, relevance, and obvious search mismatch.
- Medium-cost action → deeper content analysis and restructuring.
- Higher-cost action → substantial rewrite or specialist review.

The score helps prioritize attention, while the expected effort and potential value should be considered by the human reviewer.

The playbook therefore optimizes for useful review order rather than automatic maximum intervention.

In [3]:
# Create output directory
output_dir = "/content/work/outputs"
os.makedirs(output_dir, exist_ok=True)

# Select safe output columns
queue = work[
    [
        "rank",
        "client_hash_id",
        "content_hash_id",
        "action_score",
        "reason_code",
        "action"
    ]
].copy()

output_path = os.path.join(
    output_dir,
    "baseline_action_score.csv"
)

queue.to_csv(output_path, index=False)

print("Queue exported successfully:")
print(output_path)

print("\nRows exported:", len(queue))

display(queue.head(10))

Queue exported successfully:
/content/work/outputs/baseline_action_score.csv

Rows exported: 30000


,rank,client_hash_id,content_hash_id,action_score,reason_code,action
0,1,client_9958f0a7ae1df715,content_213eb91f21a43550,0.873086,MONITOR,Monitor
1,2,client_9958f0a7ae1df715,content_79bfcf05bc81bf82,0.872594,MONITOR,Monitor
2,3,client_9958f0a7ae1df715,content_f94fe855380e150f,0.872548,MONITOR,Monitor
3,4,client_73cda7b4e4f265ea,content_690b092cf66bc2a4,0.872539,MONITOR,Monitor
4,5,client_9958f0a7ae1df715,content_f94fe855380e150f,0.872466,MONITOR,Monitor
5,6,client_73cda7b4e4f265ea,content_690b092cf66bc2a4,0.872463,MONITOR,Monitor
6,7,client_73cda7b4e4f265ea,content_690b092cf66bc2a4,0.872351,MONITOR,Monitor
7,8,client_9958f0a7ae1df715,content_79bfcf05bc81bf82,0.872286,MONITOR,Monitor
8,9,client_9958f0a7ae1df715,content_f94fe855380e150f,0.872261,MONITOR,Monitor
9,10,client_9958f0a7ae1df715,content_f94fe855380e150f,0.872241,MONITOR,Monitor


In [4]:
import json

metrics = {
    "dataset_rows": int(len(df)),
    "queue_rows": int(len(queue)),
    "top_score": float(queue["action_score"].max()),
    "reason_code_counts": queue["reason_code"].value_counts().to_dict(),
    "note": "Directional decision-support baseline; requires human review."
}

metrics_path = os.path.join(
    output_dir,
    "w07_playbook_metrics.json"
)

with open(metrics_path, "w") as f:
    json.dump(metrics, f, indent=2)

print("Metrics saved to:")
print(metrics_path)

Metrics saved to:
/content/work/outputs/w07_playbook_metrics.json


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.